In [2]:
!pip install split-folders -q

In [3]:
import os
import zipfile
import shutil
from pathlib import Path

import splitfolders

from google.colab import drive

In [4]:
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Root project folder in Google Drive
PROJECT_DIR = "/content/drive/MyDrive/computervisionproject"

# ZIP dataset
ZIP_PATH = os.path.join(PROJECT_DIR,
                        "dataset",
                        "covid19radiography.zip")

# Temporary extraction folder
EXTRACT_DIR = "/content/COVID19_Radiography"

# Final processed dataset
PROCESSED_DIR = os.path.join(PROJECT_DIR,
                             "dataset",
                             "processed_dataset")

In [6]:
assert os.path.exists(ZIP_PATH), "Dataset ZIP not found!"

print("Dataset found!")
print(ZIP_PATH)

Dataset found!
/content/drive/MyDrive/computervisionproject/dataset/covid19radiography.zip


In [7]:
if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Dataset extracted successfully.")

Dataset extracted successfully.


In [8]:
for root, dirs, files in os.walk(EXTRACT_DIR):
    print(root)

/content/COVID19_Radiography
/content/COVID19_Radiography/COVID-19_Radiography_Dataset
/content/COVID19_Radiography/COVID-19_Radiography_Dataset/Lung_Opacity
/content/COVID19_Radiography/COVID-19_Radiography_Dataset/Lung_Opacity/masks
/content/COVID19_Radiography/COVID-19_Radiography_Dataset/Lung_Opacity/images
/content/COVID19_Radiography/COVID-19_Radiography_Dataset/COVID
/content/COVID19_Radiography/COVID-19_Radiography_Dataset/COVID/masks
/content/COVID19_Radiography/COVID-19_Radiography_Dataset/COVID/images
/content/COVID19_Radiography/COVID-19_Radiography_Dataset/Viral Pneumonia
/content/COVID19_Radiography/COVID-19_Radiography_Dataset/Viral Pneumonia/masks
/content/COVID19_Radiography/COVID-19_Radiography_Dataset/Viral Pneumonia/images
/content/COVID19_Radiography/COVID-19_Radiography_Dataset/Normal
/content/COVID19_Radiography/COVID-19_Radiography_Dataset/Normal/masks
/content/COVID19_Radiography/COVID-19_Radiography_Dataset/Normal/images


In [9]:
DATASET_ROOT = os.path.join(
    EXTRACT_DIR,
    "COVID-19_Radiography_Dataset"
)

REMOVE_CLASS = os.path.join(DATASET_ROOT,
                            "Lung_Opacity")

if os.path.exists(REMOVE_CLASS):
    shutil.rmtree(REMOVE_CLASS)

print("Remaining classes:")

print(os.listdir(DATASET_ROOT))

Remaining classes:
['Normal.metadata.xlsx', 'Lung_Opacity.metadata.xlsx', 'COVID.metadata.xlsx', 'Viral Pneumonia.metadata.xlsx', 'README.md.txt', 'COVID', 'Viral Pneumonia', 'Normal']


In [10]:
TEMP_DATASET = "/content/final_dataset"

if os.path.exists(TEMP_DATASET):
    shutil.rmtree(TEMP_DATASET)

os.makedirs(TEMP_DATASET)

classes = ["COVID","Normal","Viral Pneumonia"]

for cls in classes:

    src = os.path.join(DATASET_ROOT,
                       cls,
                       "images")

    dst = os.path.join(TEMP_DATASET,
                       cls)

    shutil.copytree(src,dst)

print("Image folders copied.")

Image folders copied.


In [11]:
for cls in classes:

    folder = os.path.join(TEMP_DATASET,
                          cls)

    count = len(os.listdir(folder))

    print(f"{cls}: {count}")

COVID: 3616
Normal: 10192
Viral Pneumonia: 1345


In [12]:
if os.path.exists(PROCESSED_DIR):
    shutil.rmtree(PROCESSED_DIR)

splitfolders.ratio(
    TEMP_DATASET,
    output=PROCESSED_DIR,
    seed=42,
    ratio=(0.8,0.1,0.1)
)

print("Dataset split completed.")

Copying files: 15153 files [04:36, 54.79 files/s]

Dataset split completed.


In [13]:
for split in ["train","val","test"]:

    print("="*50)

    print(split.upper())

    split_path = os.path.join(PROCESSED_DIR,
                              split)

    for cls in os.listdir(split_path):

        cls_path = os.path.join(split_path,
                                cls)

        print(cls,
              len(os.listdir(cls_path)))

TRAIN
COVID 2892
Viral Pneumonia 1076
Normal 8153
VAL
COVID 361
Viral Pneumonia 134
Normal 1019
TEST
COVID 363
Viral Pneumonia 135
Normal 1020


In [14]:
for root, dirs, files in os.walk(PROCESSED_DIR):

    level = root.replace(PROCESSED_DIR, "").count(os.sep)

    indent = " " * 4 * level

    print(f"{indent}{os.path.basename(root)}/")

processed_dataset/
    train/
        COVID/
        Viral Pneumonia/
        Normal/
    val/
        COVID/
        Viral Pneumonia/
        Normal/
    test/
        COVID/
        Viral Pneumonia/
        Normal/


In [15]:
print("="*60)

print("Preprocessing Completed Successfully")

print("="*60)

print(f"Saved at:\n{PROCESSED_DIR}")

Preprocessing Completed Successfully
Saved at:
/content/drive/MyDrive/computervisionproject/dataset/processed_dataset
